In [4]:
import nltk
from typing import List, Tuple, Dict
from collections import defaultdict

nltk.download('punkt', quiet=True)
from nltk.tokenize import word_tokenize


def suggest_word(prev_tokens: List[str], ngram_counts: Dict, prev_ngram_counts: Dict,
                vocabulary: set, n: int, k: float = 1.0, start_with: str = None) -> Tuple[str, float]:
    """
    Suggest most likely next word given previous tokens.

    Returns: (word, probability)
    """
    vocab_size = len(vocabulary)

    prev_ngram = tuple(prev_tokens[-(n-1):]) if len(prev_tokens) >= n-1 else tuple(['<s>'] * (n-1-len(prev_tokens)) + prev_tokens)

    best_word = None
    best_prob = 0.0

    for word in vocabulary:
        if start_with and not word.startswith(start_with):
            continue

        full_ngram = prev_ngram + (word,)
        numerator = ngram_counts.get(full_ngram, 0) + k
        denominator = prev_ngram_counts.get(prev_ngram, 0) + k * vocab_size

        prob = numerator / denominator if denominator > 0 else k / (k * vocab_size)

        if prob > best_prob:
            best_prob = prob
            best_word = word

    return best_word or '<unk>', best_prob


def get_suggestions(sentence: str, ngram_models: Dict[int, Tuple[Dict, Dict]],
                   vocabulary: set, k: float = 1.0, start_with: str = None, top_k: int = 1) -> Dict:
    """
    Get word suggestions from multiple n-gram models.

    ngram_models = {n: (ngram_counts, prev_ngram_counts), ...}

    Returns: {n: [(word, prob), ...], ...}
    """
    tokens = word_tokenize(sentence.lower())
    suggestions = {}

    for n in sorted(ngram_models.keys()):
        ngram_counts, prev_ngram_counts = ngram_models[n]
        candidates = []

        for word in vocabulary:
            if start_with and not word.startswith(start_with):
                continue

            prev_ngram = tuple(tokens[-(n-1):]) if len(tokens) >= n-1 else tuple(['<s>'] * (n-1-len(tokens)) + tokens)
            full_ngram = prev_ngram + (word,)

            numerator = ngram_counts.get(full_ngram, 0) + k
            denominator = prev_ngram_counts.get(prev_ngram, 0) + k * len(vocabulary)
            prob = numerator / denominator if denominator > 0 else k / (k * len(vocabulary))

            candidates.append((word, prob))

        candidates.sort(key=lambda x: x[1], reverse=True)
        suggestions[n] = candidates[:top_k]

    return suggestions


class AutoComplete:
    def __init__(self, ngram_models: Dict[int, Tuple[Dict, Dict]], vocabulary: set, k: float = 1.0):
        self.ngram_models = ngram_models
        self.vocabulary = vocabulary
        self.k = k

    def suggest(self, sentence: str, start_with: str = None, top_k: int = 3) -> List[str]:
        """Get top suggestions for next word."""
        suggestions = get_suggestions(sentence, self.ngram_models, self.vocabulary,
                                     self.k, start_with, top_k)

        all_suggestions = {}
        for n, words in suggestions.items():
            for word, prob in words:
                if word not in all_suggestions:
                    all_suggestions[word] = prob
                else:
                    all_suggestions[word] = max(all_suggestions[word], prob)

        ranked = sorted(all_suggestions.items(), key=lambda x: x[1], reverse=True)
        return [word for word, _ in ranked[:top_k]]